### 1. Business Problem
Late Deliveries, and orders that take longer than estimated delivery days provided might hurt the product and sellers reviews, and such recurring problems will cause customers to churn because of the unreliability of the products arrival timings. This will overall hurt the customer satisfaction. 

The primary goal of the ML models is to flag the possible late deliveries before the event happens and take action by prioritizing the fulfillment monitoring, flag the seller for follow ups, review the promised delivery estimate, escalate high-value/high-risk orders, and potentially route operational attention differently. 

### 2. Prediction Point
Prediction point will be when an order placed by customer is approved. This is when estimated delivery date will be provided and model can flag for risky orders which might have the probability of being delivered late. 

### 3. Prediction Target
The prediction target will be whether an order will be delivered after the estimated delivery date which will be flagged as risky cases. 

Specifically,
late_delivery = 1 (Risky orders, Delivered late) (Yes)

- When actual_customer_delivery_date > estimated_delivery_date

late_delivery = 0 (Non riskly, Delivered on time or earlier than estimated date) (No)

The orders that have no estimated delivery date, or orders that are processing after being approved without an estimated delivery date, will be ignored for this model. 

### 4. Eligible Training Population
All the orders that have been approved, already been processed, shipped or about to be shipped (this is unknown in this dataset), have estimated delivery date, and have been delivered. 

This provides us with proper labels whether the orders estimated delivery date beat the actual delivery date or not, meaning if the order was delivered on time, or earlier than the provided estimated delivery date.


### 5. Leakage Audit (Candidate Features)

| Column / Feature | Source | Availability | Decision | Reason |
|---|---|---|---|---|
| **From revenue_operations.silver.orders** | | | | |
| `order_approved_at` | `revenue_operations.silver.orders` | Yes | Allowed / Prediction anchor | Defines the prediction point for the model. |
| `order_estimated_delivery_date` | `revenue_operations.silver.orders` | Yes | Allowed | Estimated delivery date is known before actual delivery and defines the promised delivery SLA. |
| `purchase_day_of_week` | `revenue_operations.silver.orders` | Derived at approval | Allowed | Derived from the purchase timestamp, which is already known. |
| `purchase_hour` | `revenue_operations.silver.orders` | Derived at approval | Allowed | Derived from purchase timestamp and may capture operational timing patterns. |
| `purchase_month` | `revenue_operations.silver.orders` | Derived at approval | Allowed | Derived from purchase timestamp and may capture seasonality. |
| `estimated_delivery_days` | `revenue_operations.silver.orders` | Derived at approval | Allowed | Difference between estimated delivery date and approval/purchase time; represents how much delivery time was promised. |
| **From revenue_operations.gold.fact_orders** | | | | |
| `order_purchase_timestamp` | `revenue_operations.gold.fact_orders` | Yes | Allowed | Purchase timestamp occurs before order approval and can be used to derive time-based features. |
| `total_payment_value` | `revenue_operations.gold.fact_orders` | Yes | Allowed | Payment value is known once payment/order approval has occurred. |
| `order_delivered_customer_date` | `revenue_operations.gold.fact_orders` | No | Leakage | Actual delivery date is only known after fulfillment and directly reveals the outcome. |
| `average_review_score` | `revenue_operations.gold.fact_orders` | No | Leakage | Review score is created after fulfillment and reflects post-delivery customer satisfaction. |
| `late_delivery_flag` | `revenue_operations.gold.fact_orders` | No | Target only | This is the outcome the model is trying to predict and must never be used as an input feature. |
| **From revenue_operations.gold.fact_order_items** | | | | |
| `quantity` | `revenue_operations.gold.fact_order_items` | Derived at approval | Allowed | Quantity can be derived by counting item rows or using the Gold aggregated quantity for the order. |
| `product_subtotal` | `revenue_operations.gold.fact_order_items` | Derived at approval | Allowed | Product value is known from items in the order before delivery occurs. |
| `freight_total` | `revenue_operations.gold.fact_order_items` | Derived at approval | Allowed | Freight charges associated with the order are known before delivery. |
| **From revenue_operations.silver.order_items** | | | | |
| `shipping_limit_date` | `revenue_operations.silver.order_items` | Needs verification | Probably Allowed | Can be used only if this timestamp is already assigned and available by the chosen prediction point. |
| **From revenue_operations.silver.customers** | | | | |
| `customer_state` | `revenue_operations.silver.customers` | Yes | Allowed | Customer location for the specific order is known before/at order approval. Use the order-level customer record, not the latest Gold customer dimension. |
| `customer_city` | `revenue_operations.silver.customers` | Yes | Allowed | Customer city for the specific order is known before/at order approval. |
| **From revenue_operations.silver.payments** | | | | |
| `payment_type` | `revenue_operations.silver.payments` | Yes | Allowed | Payment method is known by approval time. Must be aggregated or encoded at the order level if multiple payment records exist. |
| `payment_installments` | `revenue_operations.silver.payments` | Yes | Allowed | Installment information is known by approval time. Must be aggregated at order level if multiple payment records exist. |
| **From revenue_operations.silver.products** | | | | |
| `product_weight_g` | `revenue_operations.silver.products` | Yes | Allowed | Product weight is known before the order is fulfilled. Must be aggregated to order level for multi-item orders. |
| `product_length_cm` | `revenue_operations.silver.products` | Yes | Allowed | Product dimensions are known before fulfillment. |
| `product_height_cm` | `revenue_operations.silver.products` | Yes | Allowed | Product dimensions are known before fulfillment. |
| `product_width_cm` | `revenue_operations.silver.products` | Yes | Allowed | Product dimensions are known before fulfillment. |
| `product_category` | `revenue_operations.silver.products` | Yes | Allowed | Product category is known when the order is placed. |
| **From revenue_operations.silver.sellers** | | | | |
| `seller_city` | `revenue_operations.silver.sellers` | Yes | Allowed | Seller location is known before fulfillment. |
| `seller_state` | `revenue_operations.silver.sellers` | Yes | Allowed | Seller location is known before fulfillment. |
| **Derived from order-item aggregations** | | | | |
| `seller_count` | order-item aggregation | Derived at approval | Allowed | Number of distinct sellers involved in an order can be calculated from order items available at approval. |
| `order_value` | order-item aggregation | Derived at approval | Allowed | Total order product value can be calculated from the items in the order before delivery. |
| `item_count` | order-item aggregation | Derived at approval | Allowed | Number of physical item rows in the order is known from the order contents. |
| `distinct_product_count` | order-item aggregation | Derived at approval | Allowed | Number of distinct products in the order can be calculated before fulfillment. |
| **Derived from order items + products** | | | | |
| `total_product_weight` | order items + products | Derived at approval | Allowed | Aggregate weight of products in the order is known from product metadata and order contents. |
| `average_product_weight` | order items + products | Derived at approval | Allowed | Average item weight can be derived from known product metadata. |
| `max_product_weight` | order items + products | Derived at approval | Allowed | Maximum product weight in the order may capture heavy-item shipping risk. |
| `total_product_volume` | order items + products | Derived at approval | Allowed | Product dimensions can be combined and aggregated to approximate shipping volume. |
| **Derived from customers + sellers + geolocation** | | | | |
| `customer_seller_distance` | customers + sellers + geolocation | Derived at approval | Allowed | Can be calculated from customer and seller locations known before fulfillment. |
| **Derived from historical orders** | | | | |
| `seller_historical_late_rate` | Historical orders + seller relationship | Historical only | Needs historical construction | For each order, calculate seller late-delivery performance using only order outcomes available before the current order approval time. |


> **Identifiers excluded from direct modeling:** `order_id`, `customer_id`, `customer_unique_id`, `product_id`, and `seller_id` will not be used directly as predictive features. They may still be retained for joins, traceability, aggregation, and construction of historical entity-level features.

> **Prediction-time rule:** A feature is only valid if it would have been available at or before `order_approved_at`. Historical features must use only information whose outcomes were known before the current order’s approval timestamp.


### 6. Success Metrics 
Precision = Among ordered predicted risky orders/ late how many were actually late?
Recall = Among actual late deliveries how many did the model predict correctly?
PR-AUC = Data maybe imbalanced so how well did the model capture both the classes late vs on time.
Confusion matrix = Classification table with TP, TN, FP, FN
Capture rate among top-risk orders = What was the rate of top riskly orders did the model predict.

### 7. Operational Constraint 

What will business actually can do differently after model says an order might be a high risk of being delivered late?

- At order approval, score every eligible order for late delivery risk. Operations reviews the highest risk 10% of orders and prioritizes those cases for early interventions. 
- Of all the orders that eventaully became late, how many were contained in the top 10& risk queue? 

Other possible use cases for business:

- Proritize high risk orders for manual review and follow ups, instead of focusing on all the risk orders, top 10% to 15% orders should be focused on by the team.
- If riskly orders are linked consistently with a single seller, following up with seller and monitoring the sellers earlier or as soon as order has been placed. 
- Most frequently selling products can be placed near local warehouses so most frequent orders not have the risk of being late. 
- High risk orders could be prioritized in warehouse handling or speed up the dispatch queue when ever possible.
- More riskly orders for delay could be used to adjust the provided estimated delivery dates. 
- Higher value orders with higher chances of being in the risk of being delivered rate may deserve more attention then lower value items. 
- If many high risk orders are concentrated in a state or city, the operations team can investigate that region logictic and handling parters. 